##  Test du modèle XGBoost Regressor avec SHAP
Ce notebook utilise les données provenant de `quotes_grouped.json`, et teste un modèle XGBoost pour prédire le prix accepté.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import train_test_split

###  Chargement des données
On charge les données et on filtre les cas où aucun prix n’a été accepté.

In [ ]:
# Chargement des données groupées par devis
df = pd.read_json("../data/quotes_grouped.json")

# On garde seulement les lignes avec prix accepté
df = df[df["proposed_price_accepted"].notnull()].copy()

# On extrait la 1ère paire de langues et 1er type de document par devis
df["source_language"] = df["languages"].apply(lambda x: x[0][0] if isinstance(x, list) and len(x) > 0 else None)
df["target_language"] = df["languages"].apply(lambda x: x[0][1] if isinstance(x, list) and len(x) > 0 else None)
df["document_type"] = df["document_types"].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)

# Conversion des catégories en codes pour XGBoost + SHAP
for col in ["source_language", "target_language", "unit", "document_type"]:
    df[col] = df[col].astype("category").cat.codes

# Test nouvelles features

df["num_languages"] = df["languages"].apply(len)
df["has_rare_doc"] = df["document_types"].apply(lambda lst: any("contract" not in d.lower() for d in lst))

# Définir les features et la cible
X = df[[
    "source_language", "target_language", "unit",
    "number_of_unit", "document_type", "requested_deadline",
    "num_languages", "has_rare_doc"
]]

# Transformation logarithmique pour réduire l'effet des valeurs extrêmes
y = np.log1p(df["proposed_price_accepted"])  # log(1 + x)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

###  Entraînement du modèle XGBoost Regressor

In [ ]:
from sklearn.model_selection import GridSearchCV

# Paramètres à tester
param_grid = {
    'n_estimators': [300, 400, 500],
    'max_depth': [6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1]
}

# Modèle de base
xgb_model = xgb.XGBRegressor(objective="reg:squarederror", random_state=42)

# GridSearch avec validation croisée (3 folds)
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_absolute_error',
    verbose=1,
    n_jobs=-1  # utilise tous les cœurs du CPU
)

# Exécution
grid_search.fit(X_train, y_train)

# Meilleurs paramètres et score
print("Meilleurs hyperparamètres :", grid_search.best_params_)
print(f"MAE moyen sur les folds : {-grid_search.best_score_:.2f} €")

# Réentraîner le meilleur modèle sur tout le jeu d'entraînement
best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

# MAE final sur le test
y_pred = best_model.predict(X_test)
#mae = mean_absolute_error(y_test, y_pred)
y_test_exp = np.expm1(y_test)
y_pred_exp = np.expm1(model.predict(X_test))
mae = mean_absolute_error(y_test_exp, y_pred_exp)
print(f"MAE final (meilleur modèle) : {mae:.2f} €")

In [ ]:
model = xgb.XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train)
print("Modèle entraîné avec succès")

### Evaluation du modèle

In [ ]:
# Moyenne des erreurs en valeur absolue (Mean Absolute Error - MAE)

#y_pred = model.predict(X_test)
#mae = mean_absolute_error(y_test, y_pred)
#print(f"MAE : {mae:.2f} €")

# Revenir à l’échelle initiale pour le calcul du MAE
y_test_exp = np.expm1(y_test)
y_pred_exp = np.expm1(model.predict(X_test))
mae = mean_absolute_error(y_test_exp, y_pred_exp)
print(f"MAE (après transformation log) : {mae:.2f} €")

In [ ]:
# MAE pondéré : on donne plus d’importance aux gros devis
mae_weighted = np.average(np.abs(y_test - y_pred), weights=y_test)
print(f"MAE pondéré : {mae_weighted:.2f} €")

In [ ]:
# Moyenne des erreurs en pourcentage (Mean Absolute Percentage Error - MAPE)

# Erreur en pourcentage moyenne
#mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
#print(f"MAPE : {mape:.2f} %")

mape = np.mean(np.abs((y_test_exp - y_pred_exp) / y_test_exp)) * 100
print(mape) 

### Interprétation avec SHAP

In [ ]:
X_train_enc = X_train.copy()
for col in X_train_enc.columns:
    if X_train_enc[col].dtype == "object":
        X_train_enc[col] = X_train_enc[col].astype("category").cat.codes
X_train_enc = X_train_enc.astype(float)

In [ ]:
explainer = shap.Explainer(model, X_train_enc)
shap_values = explainer(X_train_enc)

shap.plots.bar(shap_values, max_display=10)

In [ ]:
df["proposed_price_accepted"].hist(bins=50)
plt.title("Distribution des prix acceptés")
plt.xlabel("Prix (€)")
plt.ylabel("Nombre de devis")
plt.show()